In [ ]:
#Downloading Dataset
!wget https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log -O auth.log

--2026-03-27 00:29:45--  https://raw.githubusercontent.com/logpai/loghub/master/Linux/Linux_2k.log
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 216485 (211K) [text/plain]
Saving to: ‘auth.log’

auth.log            100%[===================>] 211.41K  --.-KB/s    in 0.02s   

2026-03-27 00:29:45 (8.27 MB/s) - ‘auth.log’ saved [216485/216485]



In [ ]:
#Loading the logs
def load_logs(file_path):
    with open(file_path, "r") as file:
        logs = file.readlines()
    return logs

logs = load_logs("auth.log")
print("Total logs:", len(logs))

Total logs: 2000


In [ ]:
#Filtering Suspicious Logs
def filter_logs(logs):
    keywords = [
        "Failed password",
        "invalid user",
        "Accepted password",
        "authentication failure",
        "sudo"
    ]
    return [log for log in logs if any(k in log for k in keywords)]

filtered_logs = filter_logs(logs)

print("Security related logs:", len(filtered_logs))

Security related logs: 490


In [ ]:
#Showing suspicious entries
print("Sample suspicious entries:\n")

for log in filtered_logs[:5]:
    print(log.strip())

Sample suspicious entries:

Jun 14 15:16:01 combo sshd(pam_unix)[19939]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=218.188.2.4
Jun 14 15:16:02 combo sshd(pam_unix)[19937]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=218.188.2.4
Jun 15 02:04:59 combo sshd(pam_unix)[20882]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root
Jun 15 02:04:59 combo sshd(pam_unix)[20884]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root
Jun 15 02:04:59 combo sshd(pam_unix)[20883]: authentication failure; logname= uid=0 euid=0 tty=NODEVssh ruser= rhost=220-135-151-1.hinet-ip.hinet.net  user=root


In [ ]:
#Install Free LLM
!pip install --upgrade transformers accelerate

In [ ]:
#Load LLM
from transformers import pipeline
import torch

generator = pipeline(
    "text-generation",
    model="tiiuae/falcon-rw-1b",
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device=0 if torch.cuda.is_available() else -1
)

print("Instruction model loaded successfully.")

In [ ]:
#Format logs
def prepare_logs_for_llm(logs, limit=10):
    selected = logs[:limit]
    formatted = "\n".join([log.strip() for log in selected])
    return formatted

In [ ]:
#Analyze logs and implement natural language explanation layer
def analyze_logs(question, logs):

    failed_logs = [log for log in logs if "authentication failure" in log or "Failed password" in log]
    success_logs = [log for log in logs if "Accepted password" in log]

    failed_count = len(failed_logs)
    success_count = len(success_logs)

    question = question.lower()

    if "summarize" in question:
        return (
            f"The logs contain {failed_count} failed login attempts and "
            f"{success_count} successful logins. "
            "Repeated authentication failures suggest possible suspicious activity."
        )

    elif "brute" in question:
        if failed_count >= 5:
            return (
                f"There are {failed_count} failed login attempts. "
                "This volume of repeated failures may indicate a brute-force attack."
            )
        else:
            return (
                f"There are {failed_count} failed login attempts. "
                "This does not strongly indicate brute-force activity."
            )

    elif "repeated" in question:
        return (
            f"There are {failed_count} failed login attempts recorded in the logs, "
            "indicating repeated authentication failures."
        )

    else:
        return (
            f"There are {failed_count} failed login attempts and "
            f"{success_count} successful login events in the logs."
        )

In [ ]:
#Query based interaction interface
def security_copilot_interface(logs):

    print("     LLM Security Copilot System   ")
    print("Ask security-related questions.")
    print("Type 'exit' to stop.\n")

    while True:
        user_query = input("Your Question: ")

        if user_query.lower() == "exit":
            print("\nExiting Security Copilot.")
            break

        response = analyze_logs(user_query, logs)

        print("\nSecurity Copilot Response:\n")
        print(response)


In [ ]:
#Run interface
security_copilot_interface(filtered_logs)

In [ ]:
#Evaluation Scenario 1
print(" Evaluation Scenario 1: High Failed Login Activity ")
print(analyze_logs("Summarize suspicious activity", filtered_logs))

 Evaluation Scenario 1: High Failed Login Activity 
The logs contain 490 failed login attempts and 0 successful logins. Repeated authentication failures suggest possible suspicious activity.


In [ ]:
#Evaluation Scenario 2
small_logs = filtered_logs[:10]

print(" Evaluation Scenario 2: Limited Failed Attempts ")
print(analyze_logs("Is there brute force behavior?", small_logs))


 Evaluation Scenario 2: Limited Failed Attempts 
There are 10 failed login attempts. This volume of repeated failures may indicate a brute-force attack.


In [ ]:
#Evaluation Scenario 3
print(" Evaluation Scenario 3: Mixed Authentication Activity")
print(analyze_logs("Summarize suspicious activity", filtered_logs))

 Evaluation Scenario 3: Mixed Authentication Activity
The logs contain 490 failed login attempts and 0 successful logins. Repeated authentication failures suggest possible suspicious activity.
